In [23]:
import pandas as pd
import numpy as np
import os

os.makedirs("../data/processed", exist_ok=True)

print("✅ Imports done, data/processed/ folder ready")

✅ Imports done, data/processed/ folder ready


####  Load all three daily raw CSVs

In [24]:
raw_files = {
    "USDINR"   : "../data/raw/USDINR_daily_raw.csv",
    "USDBRL"   : "../data/raw/USDBRL_daily_raw.csv",
    "INRBRL"   : "../data/raw/INRBRL_synthetic_raw.csv",
}

raw = {}
for name, path in raw_files.items():
    df = pd.read_csv(path)
    
    # Handle Yahoo Finance multi-row header format
    # Row 2 = "Ticker,INR=X,..." and Row 3 = "Date,,,..."
    # We detect this by checking if first cell of row 2 is "Ticker" or "Date"
    if len(df) > 0 and str(df.iloc[0, 0]).lower() in ["ticker", "date"]:
        # Re-read skipping those extra header rows
        df = pd.read_csv(path, skiprows=[1, 2])  # Skip row 1 and row 2
        print(f"{name}: {df.shape}  |  columns: {list(df.columns)}  (Yahoo Finance format)")
    else:
        print(f"{name}: {df.shape}  |  columns: {list(df.columns)}")
    
    raw[name] = df

USDINR: (1300, 6)  |  columns: ['Price', 'Close', 'High', 'Low', 'Open', 'Volume']  (Yahoo Finance format)
USDBRL: (1300, 6)  |  columns: ['Price', 'Close', 'High', 'Low', 'Open', 'Volume']  (Yahoo Finance format)
INRBRL: (1300, 4)  |  columns: ['Date', 'USDINR_Close', 'USDBRL_Close', 'INR_BRL_synthetic']


#### Standardize column names

In [25]:
def standardize_columns(df):
    """Lowercase + underscores, strip whitespace."""
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_")
          .str.replace("-", "_")
    )
    return df

standardized = {}
for name, df in raw.items():
    df = standardize_columns(df)
    print(f"\n{name} columns after standardize: {list(df.columns)}")
    standardized[name] = df


USDINR columns after standardize: ['price', 'close', 'high', 'low', 'open', 'volume']

USDBRL columns after standardize: ['price', 'close', 'high', 'low', 'open', 'volume']

INRBRL columns after standardize: ['date', 'usdinr_close', 'usdbrl_close', 'inr_brl_synthetic']


#### Parse date, set as index

In [26]:
def set_date_index(df):
    """Find the date column, parse it, set as index."""
    df = df.copy()

    # detect date column by name
    date_col = None
    for col in df.columns:
        if "date" in col.lower() or "datetime" in col.lower():
            date_col = col
            break

    # If no date column found by name, check if first column contains dates
    # (handles Yahoo Finance format where 'price' column actually has dates)
    if date_col is None:
        first_col = df.columns[0]
        # Try to parse first column as dates
        try:
            test_parse = pd.to_datetime(df[first_col], errors="coerce")
            if test_parse.notna().sum() > len(df) * 0.5:  # >50% valid dates
                date_col = first_col
                print(f"  Using '{first_col}' as date column (auto-detected)")
        except:
            pass

    if date_col is None:
        raise ValueError(f"No date column found. Columns: {list(df.columns)}")

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.set_index(date_col)
    df.index.name = "date"
    df = df.sort_index()

    print(f"  Date range: {df.index.min().date()} → {df.index.max().date()}")
    print(f"  Rows: {len(df)}")
    return df

dated = {}
for name, df in standardized.items():
    print(f"\n{name}:")
    dated[name] = set_date_index(df)


USDINR:
  Using 'price' as date column (auto-detected)
  Date range: 2021-03-04 → 2026-03-04
  Rows: 1300

USDBRL:
  Using 'price' as date column (auto-detected)
  Date range: 2021-03-04 → 2026-03-04
  Rows: 1300

INRBRL:
  Date range: 2021-03-04 → 2026-03-04
  Rows: 1300


#### Identify and select the close price column

In [41]:
def get_close_col(df, name):
    """Return the close price column name."""
    for col in df.columns:
        if "close" in col.lower():
            print(f"  {name} → close column: '{col}'")
            return col
    raise ValueError(f"No close column in {name}. Columns: {list(df.columns)}")

close_cols = {}
for name, df in dated.items():
    close_cols[name] = get_close_col(df, name)

  USDINR → close column: 'close'
  USDBRL → close column: 'close'
  INRBRL → close column: 'usdinr_close'


#### Handle missing values: forward fill small gaps only

In [28]:
def fill_small_gaps(df, close_col, max_fill_days=2, drop_threshold=3):
    """
    Forward fill gaps of <= max_fill_days.
    Drop rows where consecutive NaN gap exceeds drop_threshold.
    Returns cleaned df + a report dict.
    """
    df = df.copy()
    series = df[close_col].copy()

    original_nulls = series.isnull().sum()

    # Reindex to full calendar range (every day, not just trading days)
    full_idx = pd.date_range(start=series.index.min(),
                             end=series.index.max(),
                             freq="D")
    series_full = series.reindex(full_idx)

    # Label each NaN with its consecutive gap length
    is_null = series_full.isnull()
    gap_id  = is_null.ne(is_null.shift()).cumsum()
    gap_len = is_null.groupby(gap_id).transform("sum")

    # Forward fill only short gaps
    series_ffill = series_full.copy()
    series_ffill[is_null & (gap_len <= max_fill_days)] = np.nan  # clear short gaps first
    series_ffill = series_ffill.ffill()                           # then ffill all

    # Drop rows with long gaps (gap > drop_threshold days)
    long_gap_mask = is_null & (gap_len > drop_threshold)
    rows_to_drop  = long_gap_mask[long_gap_mask].index
    series_ffill  = series_ffill.drop(index=rows_to_drop, errors="ignore")

    # Re-align df to cleaned series index
    df_out = df.reindex(series_ffill.index).copy()
    df_out[close_col] = series_ffill

    report = {
        "original_rows"     : len(df),
        "original_nulls"    : int(original_nulls),
        "rows_after_reindex": len(full_idx),
        "long_gap_rows_dropped": len(rows_to_drop),
        "final_rows"        : len(df_out),
        "remaining_nulls"   : int(df_out[close_col].isnull().sum()),
    }
    return df_out, report

gap_reports = {}
gap_filled  = {}

for name, df in dated.items():
    ccol = close_cols[name]
    df_clean, report = fill_small_gaps(df, ccol)
    gap_filled[name]  = df_clean
    gap_reports[name] = report

    print(f"\n{'='*45}")
    print(f"  {name} — Gap Filling Report")
    print(f"{'='*45}")
    for k, v in report.items():
        print(f"  {k:<30}: {v}")


  USDINR — Gap Filling Report
  original_rows                 : 1300
  original_nulls                : 0
  rows_after_reindex            : 1827
  long_gap_rows_dropped         : 4
  final_rows                    : 1823
  remaining_nulls               : 0

  USDBRL — Gap Filling Report
  original_rows                 : 1300
  original_nulls                : 0
  rows_after_reindex            : 1827
  long_gap_rows_dropped         : 4
  final_rows                    : 1823
  remaining_nulls               : 0

  INRBRL — Gap Filling Report
  original_rows                 : 1300
  original_nulls                : 0
  rows_after_reindex            : 1827
  long_gap_rows_dropped         : 4
  final_rows                    : 1823
  remaining_nulls               : 0


#### Flag outliers with rolling 30-day window

In [29]:
def flag_outliers(df, close_col, window=30, n_std=3):
    """
    Flag rows where close is beyond rolling mean ± n_std * rolling std.
    Adds a boolean column 'is_outlier'. Does NOT delete anything.
    """
    df = df.copy()
    # For each row, it looks back at the last 30 values (window=30) and computes: The average (roll_mean) & The standard deviation (roll_std)
    roll_mean = df[close_col].rolling(window=window, min_periods=5).mean()   # min_periods=5 means it won’t calculate until at least 5 data points are available
    roll_std  = df[close_col].rolling(window=window, min_periods=5).std()

    # Standard formula
    upper = roll_mean + n_std * roll_std  
    lower = roll_mean - n_std * roll_std

    df["roll_mean"]  = roll_mean.round(4)
    df["roll_upper"] = upper.round(4)
    df["roll_lower"] = lower.round(4)
    df["is_outlier"] = (
        (df[close_col] > upper) | (df[close_col] < lower)
    )

    n_outliers = df["is_outlier"].sum()
    pct = 100 * n_outliers / len(df)
    print(f"  {close_col}: {n_outliers} outliers flagged ({pct:.2f}% of rows)")

    return df, int(n_outliers)

outlier_reports = {}
cleaned         = {}

for name, df in gap_filled.items():
    ccol = close_cols[name]
    df_out, n_out = flag_outliers(df, ccol)
    cleaned[name]         = df_out
    outlier_reports[name] = n_out

    # Show the actual outlier rows
    if n_out > 0:
        print(f"\n  Sample outlier rows for {name}:")
        print(df_out[df_out["is_outlier"]][[ccol, "roll_mean", "roll_upper", "roll_lower"]].head(5))

  close: 29 outliers flagged (1.59% of rows)

  Sample outlier rows for USDINR:
                close  roll_mean  roll_upper  roll_lower
2021-04-08  74.401497    72.8514     74.2169     71.4860
2021-06-17  73.824097    72.9607     73.7664     72.1550
2021-06-18  74.253899    72.9982     74.0689     71.9275
2021-08-30  73.496597    74.2258     74.7920     73.6597
2021-08-31  73.411201    74.1978     74.9182     73.4774
  close: 11 outliers flagged (0.60% of rows)

  Sample outlier rows for USDBRL:
             close  roll_mean  roll_upper  roll_lower
2021-06-03  5.0745     5.2729      5.4701      5.0758
2022-04-27  4.9978     4.7135      4.9583      4.4687
2023-01-04  5.4785     5.2622      5.4640      5.0603
2023-12-27  4.8135     4.9032      4.9890      4.8173
2024-04-17  5.2855     5.0428      5.2453      4.8402
  usdinr_close: 29 outliers flagged (1.59% of rows)

  Sample outlier rows for INRBRL:
            usdinr_close  roll_mean  roll_upper  roll_lower
2021-04-08     74.401497   

#### Save cleaned files

In [30]:
save_map = {
    "USDINR" : "../data/processed/USDINR_daily_clean.csv",
    "USDBRL" : "../data/processed/USDBRL_daily_clean.csv",
    "INRBRL" : "../data/processed/INRBRL_synthetic_clean.csv",
}

for name, df in cleaned.items():
    path = save_map[name]
    df.to_csv(path)
    print(f"✅ Saved {name} → {path}  ({len(df)} rows, {len(df.columns)} cols)")

✅ Saved USDINR → ../data/processed/USDINR_daily_clean.csv  (1823 rows, 9 cols)
✅ Saved USDBRL → ../data/processed/USDBRL_daily_clean.csv  (1823 rows, 9 cols)
✅ Saved INRBRL → ../data/processed/INRBRL_synthetic_clean.csv  (1823 rows, 7 cols)


#### Final summary table

In [31]:
print("\n" + "=" * 70)
print("CLEANING SUMMARY — Day 6")
print("=" * 70)
print(f"{'Dataset':<12} {'Raw Rows':>9} {'Dropped':>9} {'Final Rows':>11} {'Outliers':>10}")
print("-" * 70)

for name in cleaned:
    r        = gap_reports[name]
    raw_rows = r["original_rows"]
    dropped  = r["long_gap_rows_dropped"]
    final    = r["final_rows"]
    outliers = outlier_reports[name]
    print(f"{name:<12} {raw_rows:>9} {dropped:>9} {final:>11} {outliers:>10}")

print("=" * 70)
print("\nNote: Outliers are FLAGGED only — not deleted.")
print("Rolling window: 30 days  |  Threshold: mean ± 3 std")


CLEANING SUMMARY — Day 6
Dataset       Raw Rows   Dropped  Final Rows   Outliers
----------------------------------------------------------------------
USDINR            1300         4        1823         29
USDBRL            1300         4        1823         11
INRBRL            1300         4        1823         29

Note: Outliers are FLAGGED only — not deleted.
Rolling window: 30 days  |  Threshold: mean ± 3 std


#### Load hourly raw data

In [32]:
hourly_files = {
    "USDINR_hourly": "../data/raw/USDINR_hourly_raw.csv",
    "USDBRL_hourly": "../data/raw/USDBRL_hourly_raw.csv",
}

raw_hourly = {}
for name, path in hourly_files.items():
    df = pd.read_csv(path)
    
    # Handle Yahoo Finance multi-row header format (same as daily files)
    if len(df) > 0 and str(df.iloc[0, 0]).lower() in ["ticker", "datetime", "date"]:
        df = pd.read_csv(path, skiprows=[1, 2])
        print(f"{name}: {df.shape}  (Yahoo Finance format)")
    else:
        print(f"{name}: {df.shape}")
    
    print(f"  Columns : {list(df.columns)}")
    print(f"  First ts: {df.iloc[0, 0]}")
    print(f"  Last  ts: {df.iloc[-1, 0]}\n")
    raw_hourly[name] = df

USDINR_hourly: (10547, 6)  (Yahoo Finance format)
  Columns : ['Price', 'Close', 'High', 'Low', 'Open', 'Volume']
  First ts: 2024-03-05 01:00:00+00:00
  Last  ts: 2026-03-05 01:00:00+00:00

USDBRL_hourly: (8414, 6)  (Yahoo Finance format)
  Columns : ['Price', 'Close', 'High', 'Low', 'Open', 'Volume']
  First ts: 2024-03-05 01:00:00+00:00
  Last  ts: 2026-03-05 01:00:00+00:00



#### Standardize columns (reuse standardize_columns() function)

In [33]:
std_hourly = {}
for name, df in raw_hourly.items():
    df = standardize_columns(df)       # already defined in this notebook
    print(f"{name} columns: {list(df.columns)}")
    std_hourly[name] = df

USDINR_hourly columns: ['price', 'close', 'high', 'low', 'open', 'volume']
USDBRL_hourly columns: ['price', 'close', 'high', 'low', 'open', 'volume']


#### Timezone normalization → UTC

In [34]:
def parse_and_normalize_tz(df):
    """
    Find the datetime column, parse it, normalize to UTC, set as index.
    Handles three cases:
      1. Timestamp is tz-aware   → convert to UTC
      2. Timestamp is tz-naive   → assume UTC, localize
      3. Mixed / unparseable     → coerce, then localize
    """
    df = df.copy()

    # Detect datetime column by name
    dt_col = None
    for col in df.columns:
        if any(k in col.lower() for k in ["date", "datetime", "time"]):
            dt_col = col
            break
    
    # If no datetime column found by name, check if first column contains datetimes
    # (handles Yahoo Finance format where 'price' column actually has timestamps)
    if dt_col is None:
        first_col = df.columns[0]
        try:
            test_parse = pd.to_datetime(df[first_col], errors="coerce")
            if test_parse.notna().sum() > len(df) * 0.5:  # >50% valid dates
                dt_col = first_col
                print(f"  Using '{first_col}' as datetime column (auto-detected)")
        except:
            pass

    if dt_col is None:
        raise ValueError(f"No datetime column found. Cols: {list(df.columns)}")

    df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce", utc=False)

    # Check if tz-aware
    sample = df[dt_col].dropna().iloc[0]
    if sample.tzinfo is not None:
        # Already tz-aware — convert to UTC
        df[dt_col] = df[dt_col].dt.tz_convert("UTC")
        note = "tz-aware → converted to UTC"
    else:
        # tz-naive — assume UTC and localize
        df[dt_col] = df[dt_col].dt.tz_localize("UTC")
        note = "tz-naive → localized as UTC"

    df = df.set_index(dt_col)
    df.index.name = "datetime_utc"
    df = df.sort_index()

    print(f"  TZ note  : {note}")
    print(f"  Range    : {df.index.min()} → {df.index.max()}")
    print(f"  Rows     : {len(df)}")
    return df

tz_hourly = {}
for name, df in std_hourly.items():
    print(f"\n{name}:")
    tz_hourly[name] = parse_and_normalize_tz(df)


USDINR_hourly:
  Using 'price' as datetime column (auto-detected)
  TZ note  : tz-aware → converted to UTC
  Range    : 2024-03-05 01:00:00+00:00 → 2026-03-05 01:00:00+00:00
  Rows     : 10547

USDBRL_hourly:
  Using 'price' as datetime column (auto-detected)
  TZ note  : tz-aware → converted to UTC
  Range    : 2024-03-05 01:00:00+00:00 → 2026-03-05 01:00:00+00:00
  Rows     : 8414


#### Detect close column for hourly data

In [44]:
close_cols_hourly = {}
for name, df in tz_hourly.items():
    close_cols_hourly[name] = get_close_col(df,name)   # reuse function
    print(f"{name} → close col: '{close_cols_hourly[name]}'")

  USDINR_hourly → close column: 'close'
USDINR_hourly → close col: 'close'
  USDBRL_hourly → close column: 'close'
USDBRL_hourly → close col: 'close'


#### Forward fill hourly gaps (max 3 hours)

In [45]:
def fill_hourly_gaps(df, close_col, max_fill_hours=3, drop_threshold_hours=12):
    """
    Hourly version of gap filling.
    - Forward fill gaps <= 3 hours  (covers lunch breaks, thin markets)
    - Drop rows where consecutive gap > 12 hours (overnight/weekend handled separately)
    """
    df   = df.copy()
    series = df[close_col].copy()

    original_nulls = series.isnull().sum()

    # Reindex to full hourly range
    full_idx = pd.date_range(
        start=series.index.min(),
        end=series.index.max(),
        freq="h",
        tz="UTC"
    )
    series_full = series.reindex(full_idx)

    is_null = series_full.isnull()
    gap_id  = is_null.ne(is_null.shift()).cumsum()
    gap_len = is_null.groupby(gap_id).transform("sum")

    # Forward fill only short gaps
    series_ffill = series_full.copy()
    short_gap    = is_null & (gap_len <= max_fill_hours)
    series_ffill[short_gap] = np.nan
    series_ffill = series_ffill.ffill()

    # Drop long gap rows (weekends, holidays — expected in forex)
    long_gap_mask = is_null & (gap_len > drop_threshold_hours)
    rows_to_drop  = long_gap_mask[long_gap_mask].index
    series_ffill  = series_ffill.drop(index=rows_to_drop, errors="ignore")

    # Re-align
    df_out = df.reindex(series_ffill.index).copy()
    df_out[close_col] = series_ffill

    report = {
        "original_rows"        : len(df),
        "original_nulls"       : int(original_nulls),
        "full_hourly_range_rows": len(full_idx),
        "long_gap_rows_dropped": len(rows_to_drop),
        "final_rows"           : len(df_out),
        "remaining_nulls"      : int(df_out[close_col].isnull().sum()),
    }
    return df_out, report

gap_hourly   = {}
report_hourly = {}

for name, df in tz_hourly.items():
    ccol = close_cols_hourly[name]
    df_clean, report = fill_hourly_gaps(df, ccol)
    gap_hourly[name]    = df_clean
    report_hourly[name] = report

    print(f"\n{'='*45}")
    print(f"  {name} — Hourly Gap Report")
    print(f"{'='*45}")
    for k, v in report.items():
        print(f"  {k:<32}: {v}")


  USDINR_hourly — Hourly Gap Report
  original_rows                   : 10547
  original_nulls                  : 0
  full_hourly_range_rows          : 17521
  long_gap_rows_dropped           : 5374
  final_rows                      : 12147
  remaining_nulls                 : 0

  USDBRL_hourly — Hourly Gap Report
  original_rows                   : 8414
  original_nulls                  : 0
  full_hourly_range_rows          : 17521
  long_gap_rows_dropped           : 5363
  final_rows                      : 12158
  remaining_nulls                 : 0


#### Flag outliers (rolling 168-hour window = 1 trading week)

In [46]:
# For hourly data, 168 hours ≈ 1 week of hours
# Equivalent to the 30-day window used for daily data

outlier_hourly = {}
n_outliers_hourly = {}

for name, df in gap_hourly.items():
    ccol = close_cols_hourly[name]
    df_out, n_out = flag_outliers(df, ccol, window=168, n_std=3)  # reuse Day 6
    outlier_hourly[name]   = df_out
    n_outliers_hourly[name] = n_out

    pct = 100 * n_out / len(df_out)
    print(f"{name}: {n_out} outliers flagged ({pct:.2f}% of rows)")

    if n_out > 0:
        sample = df_out[df_out["is_outlier"]][[ccol, "roll_mean"]].head(3)
        print(f"  Sample:\n{sample}\n")

  close: 347 outliers flagged (2.86% of rows)
USDINR_hourly: 347 outliers flagged (2.86% of rows)
  Sample:
                               close  roll_mean
2024-03-07 15:00:00+00:00  82.643997    82.8213
2024-03-20 12:00:00+00:00  83.167999    82.8609
2024-03-22 04:00:00+00:00  83.367500    82.9564

  close: 267 outliers flagged (2.20% of rows)
USDBRL_hourly: 267 outliers flagged (2.20% of rows)
  Sample:
                            close  roll_mean
2024-03-05 11:00:00+00:00  4.9475     4.9452
2024-03-08 14:00:00+00:00  4.9752     4.9451
2024-03-08 17:00:00+00:00  4.9816     4.9461



#### Save hourly clean files

In [47]:
hourly_save_map = {
    "USDINR_hourly": "../data/processed/USDINR_hourly_clean.csv",
    "USDBRL_hourly": "../data/processed/USDBRL_hourly_clean.csv",
}

for name, df in outlier_hourly.items():
    path = hourly_save_map[name]
    df.to_csv(path)
    print(f"✅ Saved {name} → {path}  ({len(df)} rows)")

✅ Saved USDINR_hourly → ../data/processed/USDINR_hourly_clean.csv  (12147 rows)
✅ Saved USDBRL_hourly → ../data/processed/USDBRL_hourly_clean.csv  (12158 rows)


#### Daily vs Hourly comparison table

In [54]:
# Reload daily clean files for comparison
daily_compare = {
    "USDINR_daily": pd.read_csv("../data/processed/USDINR_daily_clean.csv",
                                 index_col=0, parse_dates=True),
    "USDBRL_daily": pd.read_csv("../data/processed/USDBRL_daily_clean.csv",
                                 index_col=0, parse_dates=True),
}

print("\n" + "=" * 78)
print("DAILY vs HOURLY — DATA QUALITY COMPARISON")
print("=" * 78)
print(f"{'Dataset':<20} {'Rows':>8} {'Nulls':>8} {'Outliers':>10} {'Outlier%':>10} {'Freq'}")
print("-" * 78)

all_compare = {}

for name, df in daily_compare.items():
    ccol = get_close_col(df, name)  # reuse existing function
    nulls = int(df[ccol].isnull().sum())
    out   = int(df["is_outlier"].sum()) if "is_outlier" in df.columns else 0
    pct   = f"{100*out/len(df):.2f}%"
    print(f"{name:<20} {len(df):>8} {nulls:>8} {out:>10} {pct:>10}   daily")

for name, df in outlier_hourly.items():
    ccol  = close_cols_hourly[name]
    nulls = int(df[ccol].isnull().sum())
    out   = int(df["is_outlier"].sum()) if "is_outlier" in df.columns else 0
    pct   = f"{100*out/len(df):.2f}%"
    print(f"{name:<20} {len(df):>8} {nulls:>8} {out:>10} {pct:>10}   hourly")

print("=" * 78)

# Ratio summary
daily_rows  = len(daily_compare["USDINR_daily"])
hourly_rows = len(outlier_hourly["USDINR_hourly"])
print(f"\nHourly/Daily row ratio  : {hourly_rows/daily_rows:.1f}x more rows in hourly data")
print(f"(Expected ~24x if market ran 24h — forex is ~17h/day active)")


DAILY vs HOURLY — DATA QUALITY COMPARISON
Dataset                  Rows    Nulls   Outliers   Outlier% Freq
------------------------------------------------------------------------------
  USDINR_daily → close column: 'close'
USDINR_daily             1823        0         29      1.59%   daily
  USDBRL_daily → close column: 'close'
USDBRL_daily             1823        0         11      0.60%   daily
USDINR_hourly           12147        0        347      2.86%   hourly
USDBRL_hourly           12158        0        267      2.20%   hourly

Hourly/Daily row ratio  : 6.7x more rows in hourly data
(Expected ~24x if market ran 24h — forex is ~17h/day active)
